Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/google-gemma/gbench/blob/main/examples/notebooks/03_golden_set_capability_smoke_tests.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench golden set functional invariants and python code verification

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook demonstrates how to verify the functional correctness of foundation models using the 16 deterministic Golden Set capability invariants in `gbench`. You will run automated smoke tests against a serving engine, understand two-sided presence matching, inspect `python_exec` unit test execution, and verify multimodal Base64 image transport.

The Golden Set spans text, code, tool/function calling, multi-turn chat templates, safety/refusal, and multimodal vision + audio. Whether every invariant is *applicable* depends on the model's declared modalities: gbench derives `supports_multimodal` / `supports_audio` from the tokenizer/checkpoint config (via `--tokenizer`), NOT from the server. Two serving paths are shown:

* **Local (Ollama) - this notebook's default:** run a quantized gemma-4 GGUF locally. A text-only GGUF serving stack makes both multimodal (vision + audio) invariants report `NOT_APPLICABLE`, so the run scores the **14** applicable text/code/tool invariants. This is the path the run cell executes.
* **Alternative (vLLM remote endpoint) - full multimodal coverage:** serve a multimodal gemma-4 checkpoint with vLLM and pass a matching `--tokenizer`. `gemma-4-26B-A4B-it` has a vision tower but **no audio tower**, so the vision invariant is applicable while the single audio invariant reports `NOT_APPLICABLE` — expect a **15/16 PASS** (14 text/code/tool + 1 vision; the audio task is N/A, neither passed nor failed). A checkpoint that also ships an audio tower would make all **16/16** applicable.

> Note: the gemma-4 GGUF (`unsloth/gemma-4-E4B-it-qat-GGUF`) and the `google/gemma-4-*` tokenizer names are the intended gemma-4 **launch artifacts** (placeholders until the model is public).

## Learning objectives

1. Serve a gemma-4 model for the Golden Set - either with vLLM (`--remote-endpoint`) or a quantized GGUF via Ollama.
2. Execute the complete Golden Set capability smoke test suite (`gbench --golden-only`).
3. Understand how two-sided presence assertions (`contains_all`, `refusal`) prevent false positive evaluation results.
4. Inspect `python_exec` unit test execution for canonical code generation tasks (runs in a bubblewrap sandbox).
5. Execute targeted capability checks using `--golden-tasks`.
6. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [gbench GitHub repository](https://www.github.com/google-gemma/gbench)
* [Ollama documentation](https://github.com/ollama/ollama)
* [Unsloth Gemma 4 QAT GGUF checkpoints](https://huggingface.co/unsloth/gemma-4-E4B-it-qat-GGUF)

## 1. Environment setup and installation

We clone the `gbench` repository from GitHub, change directory into the project root (`%cd gbench`), and install the package in editable mode (`%pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/gbench").is_dir():
        !git clone https://github.com/google-gemma/gbench.git
    %cd -q /content/gbench
else:
    if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
        if not Path("gbench").is_dir():
            !git clone https://github.com/google-gemma/gbench.git
        %cd gbench

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# The `code_canonical` golden invariant executes model-written Python inside a
# bubblewrap sandbox. gbench defaults to GBENCH_SANDBOX=required, so the
# `python_exec` task reports ERROR if the `bwrap` binary is missing. Install
# bubblewrap so the sandboxed task can run.
# (Setting GBENCH_SANDBOX=none would run model code UNSANDBOXED on this host -
# do not do that for a real smoke test; install the sandbox instead.)
!command -v bwrap >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq bubblewrap)

# Inspect available Golden Set invariant capability tests
!gbench --list golden

## Hugging Face authentication (required)

This notebook downloads the Gemma 4 GGUF (and its vision projector) plus tokenizers/datasets from the Hugging Face Hub with `huggingface_hub` — some are **gated** — so an **`HF_TOKEN` is required**.

1. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and accept the license on any gated model/dataset page you use.
2. On **Colab**: click the **🔑 key icon (Secrets)** in the left sidebar → **Add new secret**, name it `HF_TOKEN`, paste the token, and toggle **Notebook access** on.
3. **Elsewhere**: set it in your environment, e.g. `export HF_TOKEN=hf_...` (or `os.environ["HF_TOKEN"] = "hf_..."`).

The next cell loads the token and stops with instructions if it is missing.

In [ ]:
import os

# HF_TOKEN is REQUIRED: this notebook downloads the Gemma 4 GGUF (+ vision projector)
# and tokenizers/datasets from the Hugging Face Hub via huggingface_hub, which
# authenticates with it (resumable, higher rate limits, and access to gated repos).
try:
    from google.colab import userdata          # Colab: read from the Secrets vault
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass                                        # not on Colab, or the secret is unset
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_TOKEN is not set. On Colab: click the key icon (Secrets) in the left "
        "sidebar, add a secret named HF_TOKEN, and turn on notebook access. "
        "Elsewhere: os.environ['HF_TOKEN'] = 'hf_...'. "
        "Create a read token at https://huggingface.co/settings/tokens."
    )
print("HF_TOKEN loaded.")

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil, glob

# (Re)install Ollama unless BOTH the binary and its llama-server runner are present.
# A binary-only partial install fails every request with "llama-server binary not
# found", so checking only for the binary would skip the repair.
def _ollama_ready():
    if not shutil.which("ollama"):
        return False
    return any(glob.glob(p) for p in (
        "/usr/local/lib/ollama/llama-server",
        "/usr/local/lib/ollama/*/llama-server",
        "/usr/lib/ollama/llama-server",
    ))

if not _ollama_ready():
    print("Installing/repairing Ollama (binary + llama-server runner)...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama (with llama-server runner) already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Downloading the GGUF and writing the Modelfile

We download the quantized Gemma 4 GGUF **and its vision projector (`mmproj`)** from the Hugging Face Hub with `huggingface_hub` (authenticated via `HF_TOKEN`, so the download is resumable and not rate-limited), then write an Ollama `Modelfile.qat` that points `FROM` the **local** files:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

We download here rather than letting `ollama create` pull `hf.co/…` itself: Ollama's puller is anonymous (it can't use `HF_TOKEN`) and can stall on the HF CDN. The two-`FROM` import (main GGUF + `mmproj`) keeps the model's **vision** capability.

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

HF_REPO = "unsloth/gemma-4-E4B-it-qat-GGUF"
HF_QUANT = "UD-Q4_K_XL"
token = os.environ["HF_TOKEN"]  # required; loaded in the Hugging Face auth cell above

# Download the model GGUF and its vision projector (mmproj) via huggingface_hub, which
# authenticates with HF_TOKEN - Ollama's own hf.co puller is anonymous and can stall
# on the HF CDN. Build the model FROM the local files: a two-FROM import (main +
# mmproj) keeps gemma-4's vision capability (verified with `ollama show`). realpath
# resolves the HF cache symlink so `ollama create` reads the actual file.
files = HfApi().list_repo_files(HF_REPO, token=token)
main = [f for f in files if f.endswith(".gguf") and HF_QUANT in f]
proj = [f for f in files if f.endswith(".gguf") and "mmproj" in f.lower() and "-F16" in f]
if not main:
    raise RuntimeError(f"No {HF_QUANT} .gguf found in {HF_REPO}.")
GGUF_PATH = os.path.realpath(hf_hub_download(HF_REPO, main[0], token=token))
lines = [f"FROM {GGUF_PATH}"]
if proj:  # vision projector -> keeps multimodal capability
    lines.append(f"FROM {os.path.realpath(hf_hub_download(HF_REPO, proj[0], token=token))}")
lines += ['PARAMETER num_ctx 8192',
          'SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."']
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("Created Modelfile.qat from local GGUF" + (" + mmproj (vision)" if proj else ""))

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) with `ollama create -f Modelfile.qat`. Because `Modelfile.qat` points `FROM` the local GGUF (and `mmproj`) downloaded in the previous cell, this reads from disk — no network pull. We then run a quick generation test to verify the model loads into hardware memory and generates tokens correctly.

In [ ]:
import subprocess, requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from the local GGUF (built in the previous cell)...")
subprocess.run(["ollama", "create", MODEL_TAG, "-f", "Modelfile.qat"], check=True)

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in (resp.json().get("data") or [])]
print("Available REST models:", models)
if not models:
    print("No models registered yet - re-run the model registration cell above.")

## 7. Executing the 16-invariant Golden Set test suite

We execute the full capability smoke test suite using `gbench --golden-only`. The 16 mandatory invariants cover math, code generation (sandboxed `python_exec`), structured JSON output, tool/function calling, multi-turn chat-template handling, multilingual translation, exact-knowledge and entity/location identification, safety/refusal, and multimodal vision + audio.

Two serving paths are shown below:

* **Local - Ollama (this notebook's setup, runs by default).** Point `--remote-endpoint` at the Ollama endpoint from the previous cells. A text-only GGUF stack cannot satisfy the vision/audio invariants, so both report `NOT_APPLICABLE` (neither pass nor fail) and the run scores the **14** applicable text/code/tool invariants.
* **Alternative - vLLM remote endpoint (full multimodal coverage).** Serve a multimodal gemma-4 checkpoint with vLLM (e.g. `vllm serve google/gemma-4-26B-A4B-it --port 8000`) and point gbench at it with a matching `--tokenizer google/gemma-4-26B-A4B-it`. `gemma-4-26B-A4B-it` has a vision tower but no audio tower, so the vision invariant is *applicable* and the single audio invariant reports `NOT_APPLICABLE` — expect `15/16 PASS`. A checkpoint that also ships an audio tower makes all `16/16` applicable.

In [ ]:
# --- Local path: Ollama (this notebook's setup; text-only GGUF) ---
# Uses the background Ollama endpoint from the cells above. A text-only GGUF stack
# cannot satisfy the vision/audio invariants, so both report NOT_APPLICABLE and the
# 14 applicable text/code/tool invariants run.
!gbench --golden-only \
        --models gemma4-qat:4b \
        --golden-model-id gemma4-qat:4b \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_golden

# --- Alternative: vLLM (multimodal checkpoint -> the vision invariant also runs) ---
# In a separate shell:  vllm serve google/gemma-4-26B-A4B-it --port 8000
# gemma-4-26B-A4B-it has a vision tower but NO audio tower, so the vision invariant
# is applicable and the single audio invariant reports NOT_APPLICABLE -> expect
# 15/16 (16/16 for a checkpoint that also ships an audio tower).
# !gbench --golden-only \
#         --models google/gemma-4-26B-A4B-it \
#         --golden-model-id google/gemma-4-26B-A4B-it \
#         --remote-endpoint http://127.0.0.1:8000/v1 \
#         --tokenizer google/gemma-4-26B-A4B-it \
#         --results-dir ./results_golden

## 8. Listing all Golden Set capability tasks

We can list all 16 available Golden invariant tasks using `!gbench --list golden` to inspect the available task IDs, capability categories, and assertion descriptions.

In [ ]:
# List all 16 Golden Set invariant tasks
!gbench --list golden

# Run specific filtered golden tasks
!gbench --golden-only \
        --models gemma4-qat:4b \
        --golden-model-id gemma4-qat:4b \
        --golden-tasks code_canonical function_call_single structured_json \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_golden_filtered

## 9. Inspecting invariant test results

Every golden task evaluation returns strict two-sided presence matching (required terms present, forbidden terms absent) and structured tool validation to verify capability invariants without false positives.

In [ ]:
import json
from pathlib import Path

results_base = Path("./results_golden")
run_dirs = sorted([d for d in results_base.iterdir() if d.is_dir()],
                  key=lambda d: d.stat().st_mtime, reverse=True) if results_base.exists() else []
if not run_dirs:
    print("No golden results under ./results_golden - run section 7 first.")
else:
    summary_path = run_dirs[0] / "summary.json"
    golden = ([r for r in json.loads(summary_path.read_text()).get("models", [])
               if r.get("benchmark_type") == "golden"] if summary_path.exists() else [])
    if not golden:
        print("No golden result found in:", run_dirs[0])
    else:
        g = golden[0]
        acc = g.get("accuracy_percent")
        acc_str = f"{acc:.1f}%" if isinstance(acc, (int, float)) else "n/a"  # already a percent
        print(f"Model:   {g.get('model')}    status: {g.get('status')}")
        print(f"Passed:  {g.get('passed_cases')}/{g.get('applicable_tasks')} applicable "
              f"(accuracy {acc_str})")
        print(f"Also:    {g.get('not_applicable_cases')} N/A, "
              f"{g.get('failed_cases')} failed, {g.get('error_cases')} errored")
        print("-" * 66)
        marks = {"passed": "PASS", "failed": "FAIL", "error": "ERR", "not_applicable": "N/A"}
        for t in g.get("task_results", []):
            print(f"  [{marks.get(t.get('status'), str(t.get('status'))):>4}] "
                  f"{t.get('task_id')}  ({t.get('category')})")


## 10. Session cleanup and server shutdown

We terminate background Ollama server processes and remove temporary Modelfiles.

In [ ]:
import subprocess, os

subprocess.run(["pkill", "-f", "ollama"], check=False)
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")